# Mean-field and Wishart model figures

Figure 2d–e; ED Figure 3; ED Figure 8d;
and ED Figure 6a (cached main curve and static-run covariance insets).

The plot style is included. All numerical caches and simulation data must be supplied separately.
See `../README.md` for dependencies and the remaining reproduction gaps.
Code is preserved from `mean_field_figures.ipynb`; only input paths are adapted.
Stored outputs are cleared. The cached ED 6a main curve has been checked; raw static inset processing has not been executed.


In [ ]:
from pathlib import Path
import sys

_reproduction_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "src" / "reproduction_paths.py").is_file()),
    None,
)
if _reproduction_root is None:
    raise FileNotFoundError("Start Jupyter from the repository root or its notebooks folder.")
sys.path.insert(0, str(_reproduction_root / "src"))
from reproduction_paths import RAW_DATA_DIR, CACHE_DIR, MOVIE_DATA_DIR, CANN_RUNS_DIR, CANN_STATIC_RUNS_DIR, STYLE_PATH


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use(STYLE_PATH)
COLORS = {
    "FR": "#a82203",
    "CTR": "#208cc0",
    "compare": "#f1af3a",
    "Simulation": "#cf5e4e",
    "Theory": "#637b31",
    "Other": "#003967",
    "Purple": "#7B4FA3"
}

def mm_to_inch(mm):
    return mm / 25.4
def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=2.5, width=0.6)
    return ax


## Figure 2d–e: Mean-field model


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit

C_FR = 0.866
NOISE_FR = 1.22
DB_FR = 0.2              # optional FR bias shift

N = 100
T, B, SIGMA = 0.2, -0.4, 0.02
WIDTH, AMPLITUDE = 0.05 * N, 0.8
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})

neurons = np.arange(N)
distance = np.abs(neurons[:, None] - neurons[None, :])
distance = np.minimum(distance, N - distance)
W = (distance <= WIDTH).astype(float) / N
np.fill_diagonal(W, 0.0)

z, q = np.polynomial.hermite.hermgauss(128)
z, q = np.sqrt(2) * z, q / np.sqrt(np.pi)

def transfer(v, noise_gain, db=0.0):
    p = expit((v[:, None] + B + db + noise_gain * SIGMA * z) / T)
    return p @ q, (p * (1 - p)) @ q / T

def solve(x, c, noise_gain, db=0.0):
    if c <= 0 or noise_gain <= 0 or T <= 0 or SIGMA < 0:
        raise ValueError("Require c, noise_gain, T > 0 and SIGMA >= 0.")
    rho = np.full(N, 0.1)
    for _ in range(5000):
        drive = W @ rho + x
        v = c * (drive - drive.mean())
        target, gain = transfer(v, noise_gain, db)
        if np.max(np.abs(target - rho)) < 1e-10:
            return rho, gain
        rho += 0.2 * (target - rho)
    raise RuntimeError("Mean-field fixed point did not converge.")

def bump(center):
    d = (neurons - center + N / 2) % N - N / 2
    return AMPLITUDE * np.exp(-0.5 * (d / WIDTH) ** 2)

x1 = bump(N / 2)
x2 = bump(N / 2 + N * 5 / 360)   # stimulus shifted by 5 degrees

out = {}
for name, c, noise, db in [("CTR", 1.0, 1.0, 0.0),
                           ("FR", C_FR, NOISE_FR, DB_FR)]:
    mu1, gain = solve(x1, c, noise, db)
    mu2, _ = solve(x2, c, noise, db)

    chi = np.linalg.solve(np.eye(N) - c * gain[:, None] * W, np.diag(gain))
    chi = (chi + chi.T) / 2       # remove numerical asymmetry
    kappa, modes = np.linalg.eigh(chi)
    if kappa[0] <= 0:
        raise ValueError(f"{name}: susceptibility proxy is not positive definite.")

    energy_recurrent = c * np.mean(np.abs(W) @ mu1) / N

    gap_fraction = max(0.0, 1.0 - db / abs(B))  # requires B < 0

    out[name] = dict(
        mu1=mu1, mu2=mu2, dmu=mu1 - mu2, c=c,
        chi=chi, kappa=kappa, modes=modes,
        energy=energy_recurrent * gap_fraction,
    )


### Figure 2d: Changes in noise correlation, dimensionality, and energy proxy


In [ ]:
from matplotlib.ticker import MaxNLocator

metrics = {}
for name, data in out.items():
    covariance = T * data["chi"]
    std = np.sqrt(np.diag(covariance))
    correlation = covariance / np.outer(std, std)
    mean_correlation = (
        correlation.sum() - np.trace(correlation)
    ) / (N * (N - 1))

    eigenvalues = T * data["kappa"]
    dimension = eigenvalues.sum() ** 2 / np.sum(eigenvalues ** 2)
    metrics[name] = np.array([
        mean_correlation, dimension, data["energy"]
    ])

if np.any(np.abs(metrics["CTR"]) < 1e-14):
    raise ValueError("Percentage change is undefined for a zero CTR metric.")

changes = 100 * (metrics["FR"] / metrics["CTR"] - 1)

with plt.style.context(STYLE_PATH):
    fig, ax = plt.subplots(
        figsize=(40 / 25.4, 35 / 25.4), dpi=500
    )

    ax.bar(
        np.arange(3), changes,
        width=0.55, color="#f1af3a",
        edgecolor="black", linewidth=0.4, zorder=3,
    )
    ax.axhline(
        0, color="black", lw=0.6,
        ls=(0, (2, 2)), zorder=2,
    )

    ax.set_xticks(
        np.arange(3),
        [r"$\langle C_{ij}\rangle$",
         r"$D_{\mathrm{PR}}$",
         r"$E_{\mathrm{proxy}}$"],
    )
    ax.set_ylabel("FR SNR relative to CTR (%)")  # percentage change
    ax.set_xlim(-0.55, 2.45)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(direction="out", length=2.5, width=0.6)

    fig.tight_layout(pad=0.3)
    plt.show()


### Figure 2e: Noise orientation and signal overlap by covariance mode


In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator
from matplotlib.ticker import ScalarFormatter

sigma_xi = 0.5
input_power_per_mode = np.sum((x1 - x2) ** 2) / N
mode_results = {}

for name, data in out.items():
    kappa = data["kappa"]
    modes = data["modes"]
    dmu = data["dmu"]
    c = data["c"]

    eigenvalues = (c * sigma_xi * kappa) ** 2
    signal_power = np.dot(dmu, dmu)
    if signal_power == 0 or input_power_per_mode == 0:
        raise ValueError("Cos² requires a nonzero stimulus and response difference.")

    overlap = (modes.T @ dmu) ** 2
    ideal_overlap = input_power_per_mode * (c * kappa) ** 2

    mode_results[name] = dict(
        eigenvalues=eigenvalues,
        overlap=overlap,
        cos2=overlap / signal_power,
        ideal_overlap=ideal_overlap,
        ideal_cos2=ideal_overlap / ideal_overlap.sum(),
    )

with plt.style.context(STYLE_PATH):
    fig, axes = plt.subplots(
        1, 2,
        figsize=(80 / 25.4, 38 / 25.4),
        dpi=500, constrained_layout=True,
    )

    for name in ("FR", "CTR"):
        data = mode_results[name]
        lam = data["eigenvalues"]

        for ax, key in zip(axes, ("cos2", "overlap")):

            ax.plot(
                np.r_[0, lam],
                np.r_[0, data["ideal_" + key]],
                color=COLORS[name], lw=1,
                label=name, zorder=3,
                marker="o", markersize=2, alpha=0.7,
            )

        for ax in axes:
                ax.set_xlabel(r"$\lambda_k$")
                ax.set_ylim(bottom=0)
                ax.spines[["top", "right"]].set_visible(False)
                ax.tick_params(direction="out", length=2.5, width=0.6)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
                ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
                ax.autoscale(enable=True, axis="both")
                ax.margins(x=0.05, y=0.05)

    axes[0].set_ylabel(r"$\cos^2(\phi_k)$")
    axes[1].set_ylabel(
        r"$(\mathbf{x}_k\cdot\mathbf{\Delta\mu})^2$"
    )

    axes[1].legend(
        frameon=False, handletextpad=0.4,
        borderaxespad=0.2, loc="upper left",
    )

    plt.show()


## Extended Data Figure 3: Wishart model

Uses `jun8_overlap_cos2_no_top_mode.npz`, which contains the original 39-mode
FR/CTR eigenvalue, overlap, alignment, and SEM arrays. Supply this file in `CACHE_DIR` (default: `data/cache/` relative to the repository root).
The Wishart-generation code belongs to a separate paper and is outside this release's scope.


### ED Figure 3a: Signal overlap versus covariance eigenvalue


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.style.use(STYLE_PATH)

LINE_STYLE = dict(
    lw=0.9,
    marker="o",
    markersize=2,
    markeredgecolor="black",
    markeredgewidth=0.25,
    zorder=3,
)

def format_panel(ax, ylabel):
    ax.set_xlabel(
        r"$\langle \lambda_k \rangle$",
        fontsize=7, labelpad=1,
    )
    if ylabel == 'Overlap':
        ax.set_ylabel(
            r"$\langle (\boldsymbol{\nu}_k \cdot \boldsymbol{\Delta\mu})^2 \rangle$",
            fontsize=7, labelpad=1,
        )
    else:
        ax.set_ylabel(ylabel, fontsize=7, labelpad=1)

    ax.spines[["top", "right"]].set_visible(False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    ax.tick_params(
        axis="both", which="major",
        direction="out",
        labelsize=6, length=2, width=0.5, pad=1.5,
    )
    ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    ax.margins(x=0.05)
    ax.grid(False)

with np.load((CACHE_DIR / 'jun8_overlap_cos2_no_top_mode.npz')) as data:
    x_fr = data["fr_eigenvalue"]
    overlap_fr = data["fr_overlap"]
    overlap_sem_fr = data["fr_overlap_sem"]

    x_ctr = data["ctr_eigenvalue"]
    overlap_ctr = data["ctr_overlap"]
    overlap_sem_ctr = data["ctr_overlap_sem"]

fig, ax = plt.subplots(figsize=(1.5, 1.2), dpi=300)

for label, x, y, sem in [
    ("FR", x_fr, overlap_fr, overlap_sem_fr),
    ("CTR", x_ctr, overlap_ctr, overlap_sem_ctr),
]:
    ax.fill_between(
        x, y - sem, y + sem,
        color=COLORS[label],
        alpha=0.2, linewidth=0, zorder=1,
    )
    ax.plot(x, y, color=COLORS[label], label=label, **LINE_STYLE)

format_panel(ax, "Overlap")
ax.legend(
    frameon=False, fontsize=6, loc="best",
    handlelength=1.2, handletextpad=0.4,
    labelspacing=0.2, borderaxespad=0.3,
)

fig.tight_layout(pad=0.3)
plt.show()


### ED Figure 3b: Cumulative SNR over covariance modes


In [ ]:
with np.load((CACHE_DIR / 'jun8_overlap_cos2_no_top_mode.npz')) as data:
    x_fr = data["fr_eigenvalue"]
    overlap_fr = data["fr_overlap"]
    overlap_sem_fr = data["fr_overlap_sem"]

    x_ctr = data["ctr_eigenvalue"]
    overlap_ctr = data["ctr_overlap"]
    overlap_sem_ctr = data["ctr_overlap_sem"]

fr_order = np.argsort(x_fr)[::-1]
x_fr = x_fr[fr_order]
overlap_fr = overlap_fr[fr_order]
overlap_sem_fr = overlap_sem_fr[fr_order]

ctr_order = np.argsort(x_ctr)[::-1]
x_ctr = x_ctr[ctr_order]
overlap_ctr = overlap_ctr[ctr_order]
overlap_sem_ctr = overlap_sem_ctr[ctr_order]

snr_fr = overlap_fr / x_fr
snr_ctr = overlap_ctr / x_ctr

snr_sem_fr = overlap_sem_fr / x_fr
snr_sem_ctr = overlap_sem_ctr / x_ctr

cumsum_snr_fr = np.cumsum(snr_fr)
cumsum_snr_ctr = np.cumsum(snr_ctr)

cumsum_snr_sem_fr = np.sqrt(np.cumsum(snr_sem_fr**2))
cumsum_snr_sem_ctr = np.sqrt(np.cumsum(snr_sem_ctr**2))

modes_fr = np.arange(1, len(cumsum_snr_fr) + 1)
modes_ctr = np.arange(1, len(cumsum_snr_ctr) + 1)

fig, ax = plt.subplots(figsize=(1.5, 1.2), dpi=300)

for label, x, y, sem in [
    ("FR", modes_fr, cumsum_snr_fr, cumsum_snr_sem_fr),
    ("CTR", modes_ctr, cumsum_snr_ctr, cumsum_snr_sem_ctr),
]:
    ax.fill_between(
        x, y - sem, y + sem,
        color=COLORS[label],
        alpha=0.2, linewidth=0, zorder=1,
    )
    ax.plot(
        x, y,
        color=COLORS[label],
        label=label,
        **LINE_STYLE,
    )
    fit = np.polyfit(modes_fr[:9], cumsum_snr_fr[:9], 1)
ax.plot(modes_fr, np.polyval(fit, modes_fr),
        color="grey", linestyle="--", linewidth=1)

format_panel(ax, "Cumulative SNR")
ax.set_xlabel("Number of modes", fontsize=7, labelpad=1)
ax.xaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))

ax.legend(
    frameon=False, fontsize=6, loc="best",
    handlelength=1.2, handletextpad=0.4,
    labelspacing=0.2, borderaxespad=0.3,
)

fig.tight_layout(pad=0.3)
plt.show()


## Extended Data Figure 6a: Averaging-window validation

Code from `figure_producer_static.ipynb`. The main SNR curve loads the saved
`data/cache/static_snr_ratio_FR_over_CTR_data.npz`, avoiding the slow raw-data
SNR sweep. The original six-point selection, sign convention, and conversion of
averaging size to milliseconds are retained. Supply the NPZ separately in `CACHE_DIR`. The slow raw-data SNR sweep
is outside this notebook's default plotting path.


### ED Figure 6a main: SNR difference versus averaging duration — saved data


In [ ]:
from pathlib import Path

data_name = 'static_snr_ratio_FR_over_CTR_data.npz'
snr_out = Path(CACHE_DIR)
data = np.load(snr_out / data_name)
print("Loaded data from", data_name)
x = data['average_size']
y = data['delta_snr_fr_over_ctr_percent']
from matplotlib.ticker import MaxNLocator


In [ ]:
compare_color = COLORS.get("compare", "#f1af3a")
y_final = y[:-3]
x_final = 3*x[:-3]
fig, ax = plt.subplots(
    figsize=(mm_to_inch(30), mm_to_inch(35)),
    dpi=500,
)

ax.plot(
    x_final,
    -y_final,
    "o-",
    color=compare_color,
    lw=1.0,
    ms=3,
)

ax.axhline(
    0,
    color="0.4",
    lw=0.6,
    linestyle=":",
    zorder=0,
)

ax.set_xlabel(r"Averaging (ms)", fontsize=6)
ax.set_ylabel("CTR SNR relative to FR (%)", fontsize=6)
ax.xaxis.set_major_locator(MaxNLocator(4))
ax.yaxis.set_major_locator(MaxNLocator(4))
ax.tick_params(axis="both", which="major", labelsize=6)

clean_axis(ax)
fig.tight_layout(pad=0.3)

plt.show()


### ED Figure 6a insets: Covariance validation at 75 ms and 300 ms

The original streaming loader and covariance/linear-response calculations are
copied from source cell 6, with `AVERAGE_SIZES = [25, 100]`. The plot's conversion
of three milliseconds per size gives the two inset durations.

**Not executed:** these cells require the original static run folders under
`data/continuous_attractor_static/` (override with `CALCIUM_STATIC_RUNS_DIR`), including each run's `run_metadata.json`
and `state_history.npy`. They do not run a new simulation, but reading and
processing the state histories may be slow. Run the main cached plot above
independently if those run data are unavailable.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np

RUNS_DIR = Path(CANN_STATIC_RUNS_DIR)
AMPAR = 1.0
RIN = 1.0
TEMP = 0.02
ETA = 0.02
THETA = 0.01
IDIR = 0.5

AVERAGE_SIZES = [25, 100]

def close(a, b, tol=1e-12):
    return math.isclose(float(a), float(b), rel_tol=tol, abs_tol=tol)

def first_not_none(*values):
    for value in values:
        if value is not None:
            return value
    return None

def get_recurrent_width(spec, net):
    return first_not_none(
        spec.get("cann_width"),
        net.get("recurrent_width_fraction"),
        spec.get("threshold"),
        net.get("threshold_active_fraction"),
    )

def matching_state_files(runs_dir, *, ampar, rin, temp, eta, theta, idir):
    files = []
    recurrent_widths = []
    block_sizes = []

    for meta_path in sorted(runs_dir.glob("*/run_metadata.json")):
        meta = json.loads(meta_path.read_text())
        spec = meta.get("spec", {})
        net = meta.get("network_params", {})

        if not close(spec.get("ampar"), ampar):
            continue
        if not close(spec.get("rin"), rin):
            continue
        if not close(spec.get("sigma_temp"), temp):
            continue
        if not close(spec.get("sigma_eta"), eta):
            continue
        if not close(spec.get("sigma_theta"), theta):
            continue
        if not close(spec.get("idir"), idir):
            continue

        state_path = meta_path.parent / "state_history.npy"
        if state_path.exists():
            files.append(state_path)
            recurrent_widths.append(get_recurrent_width(spec, net))
            block_sizes.append(spec.get("block_size"))

    if not files:
        raise FileNotFoundError("No matching state_history.npy files found.")

    recurrent_widths = sorted({x for x in recurrent_widths if x is not None})
    block_sizes = sorted({x for x in block_sizes if x is not None})

    if len(recurrent_widths) != 1:
        raise ValueError(f"Expected one recurrent width, found {recurrent_widths}")
    if len(block_sizes) != 1:
        raise ValueError(f"Expected one block size, found {block_sizes}")

    return files, int(block_sizes[0]), float(recurrent_widths[0])

def streamed_windowed_covariance(state_files, average_size, block_chunk=20_000):
    n = 0
    sum_x = None
    sum_xx = None

    for path in state_files:
        arr = np.load(path, mmap_mode="r")  # generations x neurons
        generations, neurons = arr.shape
        n_blocks = generations // average_size

        if sum_x is None:
            sum_x = np.zeros(neurons, dtype=np.float64)
            sum_xx = np.zeros((neurons, neurons), dtype=np.float64)

        for b0 in range(0, n_blocks, block_chunk):
            b1 = min(b0 + block_chunk, n_blocks)

            rows = arr[b0 * average_size : b1 * average_size]
            X = rows.reshape(b1 - b0, average_size, neurons).mean(axis=1)

            sum_x += X.sum(axis=0)
            sum_xx += X.T @ X
            n += X.shape[0]

    mean = sum_x / n
    covariance = (sum_xx - n * np.outer(mean, mean)) / (n - 1)
    return covariance, mean, n

def calculate_R_from_mean(mean_firing_rate, N, T, ampar, rin, recurrent_width, yes_t=True):
    idx = np.arange(N)
    diff = np.abs(idx[:, None] - idx[None, :])
    diff = np.minimum(diff, N - diff)

    threshold = recurrent_width * N / 2.0
    weights = (diff <= threshold).astype(float) / N
    np.fill_diagonal(weights, 0.0)

    D = np.diag(mean_firing_rate * (1.0 - mean_firing_rate))
    if yes_t:
        D = D / T

    return np.linalg.solve(np.eye(N) - rin * ampar * D @ weights, D)

state_files, BLOCK_SIZE, RECURRENT_WIDTH = matching_state_files(
    RUNS_DIR,
    ampar=AMPAR,
    rin=RIN,
    temp=TEMP,
    eta=ETA,
    theta=THETA,
    idir=IDIR,
)

average_width_results = {}

for average_size in AVERAGE_SIZES:
    covariance_full, mean_firing_rate, n_samples = streamed_windowed_covariance(
        state_files,
        average_size=average_size,
    )

    eigenvalues_full = np.linalg.eigvalsh(covariance_full)

    N = mean_firing_rate.shape[0]
    Gamma = calculate_R_from_mean(
        mean_firing_rate,
        N,
        TEMP,
        AMPAR,
        RIN,
        RECURRENT_WIDTH,
        yes_t=True,
    )

    tau_gamma = BLOCK_SIZE
    f_T = 1.0 / average_size
    fac_gamma = min(tau_gamma, average_size) / average_size

    gamma_eval = np.linalg.eigvalsh(Gamma)

    pred_linear = np.linalg.eigvalsh(f_T * TEMP * Gamma)
    pred_quadratic = np.linalg.eigvalsh(
        fac_gamma * (THETA * AMPAR * RIN) ** 2 * (Gamma @ Gamma.T)
    )
    pred_full = pred_linear + pred_quadratic

    order = np.argsort(gamma_eval)

    average_width_results[average_size] = {
        "gamma_eval": gamma_eval[order],
        "simulation": eigenvalues_full[order],
        "pred_linear": pred_linear[order],
        "pred_quadratic": pred_quadratic[order],
        "pred_full": pred_full[order],
        "n_samples": n_samples,
    }

print("Computed average widths:", list(average_width_results))


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.style.use(STYLE_PATH)

COLORS = globals().get("COLORS", {
    "Simulation": "#cf5e4e",
    "Other": "#003967",
})

def mm_to_inch(mm):
    return mm / 25.4

def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=2.5, width=0.6)
    return ax

for average_size, result in average_width_results.items():
    fig, ax = plt.subplots(
        figsize=(mm_to_inch(25), mm_to_inch(25)),
        dpi=500,
        constrained_layout=False,
    )

    ax.plot(
        result["gamma_eval"],
        result["pred_full"],
        "-",
        color="#003967",
        lw=1.5,
        label=r"Full theory",
        zorder=2,
    )

    ax.plot(
        result["gamma_eval"],
        result["pred_quadratic"],
        "--",
        color="0.45",
        lw=1.0,
        label=r"Q theory",
        zorder=1,
    )

    ax.plot(
        result["gamma_eval"],
        result["simulation"],
        "o",
        color="#cf5e4e",
        ms=1.8,
        markeredgewidth=0,
        label="Simulation",
        zorder=3,
    )

    ax.set_title(rf"$L={average_size*3}$")
    ax.set_xlabel(r"$z_i$")
    ax.set_ylabel(r"$\lambda_i$")

    ax.set_xlim(3.9, 8.9)
    ax.set_ylim(0.001, 0.01)
    if average_size == AVERAGE_SIZES[0]:
        ax.set_ylim(0.001, 0.015)

    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_locator(MaxNLocator(4))

    ax.legend(
        frameon=False,
        loc="upper left",
        handlelength=1.8,
        borderaxespad=0.2,
    )

    clean_axis(ax)
    fig.tight_layout(pad=0.3)

    plt.show()


## Extended Data Figure 8d: SNR per energy proxy and linear-response inset


In [ ]:
import numpy as np
from matplotlib import pyplot as plt

gamma_r = np.linspace(0, 3, 500)

def snr_per_energy(gamma_r, c, k=1):
    x = c * gamma_r
    z = 1 / (4 * np.cosh(x / 2)**2)
    return x * z / (c**2 * z + k)

figure, ax = plt.subplots(figsize=(mm_to_inch(40), mm_to_inch(35)))

for label, c, fallback in [("CTR", 1.0, "#208cc0"), ("FR", 0.86, "#a82203")]:
    color = COLORS.get(label, fallback)
    efficiency = snr_per_energy(gamma_r, c)
    max_index = np.nanargmax(efficiency)
    ax.plot(gamma_r, efficiency, linewidth=1.5, color=color, label=label)
    ax.scatter(gamma_r[max_index], efficiency[max_index],
               color="black", s=20, zorder=3)
    ax.axvline(gamma_r[max_index], color=color, linestyle="--", linewidth=1)

ax.set_xlabel(r"$\gamma_r$", fontsize=16)
ax.set_ylabel(r"SNR / E$_{\mathrm{Proxy}}$", fontsize=12)
ax.tick_params(axis="both", which="major", labelsize=12)
ax.set_xlim(0.8, 2.5)
ax.set_ylim(0.13, 0.23)
ax.spines[["top", "right"]].set_visible(False)

inset = ax.inset_axes([0.12, 0.69, 0.30, 0.27])
x = np.linspace(0, 5, 200)
z = 1 / (4 * np.cosh(x / 2)**2)
inset.plot(x,z, linewidth=1.2, color=COLORS.get("Theory", "#637b31"))
inset.set( ylim=(0, 0.26), xticks=[], yticks=[])
inset.set_ylabel(r"$z$", fontsize=7, labelpad=0)
inset.spines[["top", "right"]].set_visible(False)

plt.show()
